In [4]:
# importando os scripts de consulta

import sys
import os

# Altera o caminho para a pasta raiz do projeto (um nível acima)
raiz_projeto = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Adiciona a raiz do projeto ao caminho de busca do Python se ainda não estiver lá
if raiz_projeto not in sys.path:
    sys.path.append(raiz_projeto)

# Agora você pode importar os módulos da pasta scripts usando a notação de ponto
from scripts.crm_leads import load_leads
from scripts.crm_inscricoes import load_inscricoes





In [34]:
leads = load_leads()


Página 1: 250 registros
Página 2: 250 registros
Página 3: 250 registros
Página 4: 250 registros
Página 5: 250 registros
Página 6: 250 registros
Página 7: 250 registros
Página 8: 250 registros
Página 9: 250 registros
Página 10: 250 registros
Página 11: 250 registros
Página 12: 250 registros
Página 13: 250 registros
Página 14: 250 registros
Página 15: 250 registros
Página 16: 250 registros
Página 17: 250 registros
Página 18: 250 registros
Página 19: 250 registros
Página 20: 250 registros
Página 21: 250 registros
Página 22: 250 registros
Página 23: 250 registros
Página 24: 250 registros
Página 25: 250 registros
Página 26: 250 registros
Página 27: 250 registros
Página 28: 250 registros
Página 29: 250 registros
Página 30: 250 registros
Página 31: 250 registros
Página 32: 250 registros
Página 33: 250 registros
Página 34: 250 registros
Página 35: 250 registros
Página 36: 250 registros
Página 37: 250 registros
Página 38: 250 registros
Página 39: 250 registros
Página 40: 250 registros
Página 41

In [6]:
"""
CRM Educacional -> Power BI
Script refatorado (template)

Preencha a lista `atributos` e o dicionário `rename_columns`
com os valores do seu projeto.
"""

import time
import requests
import xmltodict
import pandas as pd

BASE_URL = "https://trivento.api.crmeducacional.com"
URL = f"{BASE_URL}/api/LocalizacaoAvancada"

USERNAME = "IntegraçãoCRM"
PASSWORD = "IntegraçãoCRM"
AUTH_TYPE = "APICRMEducacional"

atributos = [
    'cad_numeroinscricao',
    "cad_clientepotencialid",
    "crmeduc_data_aprovacao",
    "cad_datapagamento",
    "cad_situacaoavaliacao",
    "cad_situacao",
    "cad_ofertaid",
    "trivento_unidadedeensinotrivento",
    "cad_concursoid",
    "coc_situacaopagamento",
]

rename_columns = {
    'cad_inscricaocandidatoid': 'id_inscricao',
    "cad_clientepotencialid": "ID Lead",
    'cad_numeroinscricao': 'Numero Inscricao',
    "crmeduc_data_aprovacao": "Data Aprovação",
    "cad_datapagamento": "Data de Pagamento",
    "cad_situacaoavaliacao": "Situação de Avaliação",
    "cad_situacao": "Situação Inscrição",
    "cad_ofertaid": "Curso ID",
    "trivento_unidadedeensinotrivento": "Unidade Inscrito",
    "cad_concursoid": "Forma de Ingresso",
    "coc_situacaopagamento": "Situação Pagamento",
}


def get_token(session):
    payload = {
        "grant_type": "password",
        "username": USERNAME,
        "password": PASSWORD,
        "authentication_type": AUTH_TYPE,
    }

    response = session.post(
        f"{BASE_URL}/api/token",
        data=payload,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()["access_token"]


def create_headers(token):
    return {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/xml",
    }


def create_fetch_xml(attributes):
    tags = "\n".join(
        f'<attribute name="{c}"/>' for c in attributes
    )

    return f"""
<fetch version="1.0" output-format="xml-platform" mapping="logical">
  <entity name="cad_inscricaocandidato">
    {tags}
    <order attribute="createdon" descending="true"/>
  </entity>
</fetch>
"""


def post_retry(session, *args, retries=3, **kwargs):
    for i in range(retries):
        try:
            r = session.post(*args, **kwargs)
            r.raise_for_status()
            return r
        except requests.RequestException:
            if i == retries - 1:
                raise
            time.sleep(5)


def get_page(session, page, headers, body):
    headers = headers.copy()
    headers["Pagina"] = str(page)

    response = post_retry(
        session,
        URL,
        headers=headers,
        data=body.encode("utf-8"),
        timeout=60,
    )

    doc = xmltodict.parse(response.text)

    resultset = doc["resultset"]
    records = resultset["result"]

    if isinstance(records, dict):
        records = [records]

    return records, resultset["@morerecords"] == "1"


def get_all_records(session, headers, body):
    page = 1
    rows = []

    while True:
        records, more = get_page(session, page, headers, body)
        rows.extend(records)
        print(f"Página {page}: {len(records)} registros")

        if not more:
            break

        page += 1

    return rows


def flatten_value(value):
    if not isinstance(value, dict):
        return value

    if "@name" in value:
        return value["@name"]

    if "#text" in value:
        return value["#text"]

    return value


def flatten_record(record):
    return {
        k: v['#text'] if k == 'cad_clientepotencialid' else  flatten_value(v)
        for k, v in record.items()
    }


def load_inscricoes():

    with requests.Session() as session:

        token = get_token(session)
        headers = create_headers(token)
        body = create_fetch_xml(atributos)

        records = get_all_records(
            session,
            headers,
            body,
        )
    
    df = pd.DataFrame(
        flatten_record(r)
        for r in records
    )

    for col in ["'createdon'", "modifiedon"]:
        if col in df.columns:
            df[col] = pd.to_datetime(
                df[col],
                format="%d/%m/%Y %H:%M",
                errors="coerce",
            )

    df.rename(
        columns={
            k: v
            for k, v in rename_columns.items()
            if k in df.columns
        },
        inplace=True,
    )

    map = {
        "Faculdade Serra Dourada de " : "",
        "Faculdade Alis" : "Itabirito",
        " dos Carajás" : ""}
    
    df['Unidade Inscrito'] = df['Unidade Inscrito'].map(map)
    return df

inscricoes = load_inscricoes()





Página 1: 250 registros
Página 2: 250 registros
Página 3: 250 registros
Página 4: 250 registros
Página 5: 250 registros
Página 6: 250 registros
Página 7: 250 registros
Página 8: 250 registros
Página 9: 250 registros
Página 10: 250 registros
Página 11: 250 registros
Página 12: 250 registros
Página 13: 250 registros
Página 14: 250 registros
Página 15: 250 registros
Página 16: 250 registros
Página 17: 250 registros
Página 18: 250 registros
Página 19: 250 registros
Página 20: 250 registros
Página 21: 250 registros
Página 22: 250 registros
Página 23: 234 registros


In [7]:
print(f'Total de leads: {leads.shape[0]}')
print(f'Total de inscricoes: {inscricoes.shape[0]}')

Total de leads: 19623
Total de inscricoes: 5734


In [8]:
leads.columns

Index(['Tabulação Realizada', 'Nome Completo', 'Nome', 'Sobrenome', 'Email',
       'Celular', 'Telefone', 'ID Lead', 'Data Criação', 'Unidade Form',
       'Situação Funil', 'Origem Lead', 'Proprietário', 'Canal',
       'Inscrição Atual', 'cpf', 'Última Tabulação', 'Idade', 'Concurso Atual',
       'Concurso de Origem', 'Data Nascimento', 'Oferta Atual', 'Unidade Lead',
       'Linha de Negócio', 'Nacionalidade', 'Estado Civil', 'Sexo',
       'Órgão Expedidor', 'RG', 'Cor/Raça', 'Tabulação Lead',
       'Detalhamento Tabulação'],
      dtype='str')

In [35]:
# filtrando os leads sem nenhum dado

colunas = ['Nome Completo', 'Nome', 'Sobrenome', 'Email',
       'Celular', 'Telefone']

linhas_em_branco = leads[leads[colunas].isna().all(axis=1)]

In [36]:
linhas_em_branco.sample(10)

,Inscrição Atual,cpf,Tabulação Realizada,Nome Completo,Nome,Sobrenome,Email,Celular,ID Lead,Data Criação,...,Unidade Lead,Última Tabulação,Nacionalidade,Estado Civil,Sexo,Órgão Expedidor,RG,Cor/Raça,Tabulação Lead,Detalhamento Tabulação
4378,NaN,NaN,Não,NaN,NaN,NaN,NaN,NaN,{08C2B300-CBB2-F111-AAAB-002248E10B2D},NaT,...,NaN,NaN,Brasil,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2712,NaN,NaN,Não,NaN,NaN,NaN,NaN,NaN,{E95ACB37-CCB2-F111-AAAB-002248E10B2D},NaT,...,NaN,NaN,Brasil,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6078,NaN,NaN,Não,NaN,NaN,NaN,NaN,NaN,{2B90B975-C9B2-F111-AAAB-002248E10B2D},NaT,...,NaN,NaN,Brasil,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6338,NaN,NaN,Não,NaN,NaN,NaN,NaN,NaN,{0DC8DD3F-C9B2-F111-AAAB-002248E10B2D},NaT,...,NaN,NaN,Brasil,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2362,NaN,NaN,Não,NaN,NaN,NaN,NaN,NaN,{F23BB56D-CCB2-F111-AAAB-002248E10B2D},NaT,...,NaN,NaN,Brasil,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6920,NaN,NaN,Não,NaN,NaN,NaN,NaN,NaN,{085912C8-C8B2-F111-AAAB-002248E10B2D},NaT,...,NaN,NaN,Brasil,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1798,NaN,NaN,Não,NaN,NaN,NaN,NaN,NaN,{468D88BB-CCB2-F111-AAAB-002248E10B2D},NaT,...,NaN,NaN,Brasil,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3371,NaN,NaN,Não,NaN,NaN,NaN,NaN,NaN,{8CDF02CC-CBB2-F111-AAAB-002248E10B2D},NaT,...,NaN,NaN,Brasil,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3765,NaN,NaN,Não,NaN,NaN,NaN,NaN,NaN,{24802F90-CBB2-F111-AAAB-002248E10B2D},NaT,...,NaN,NaN,Brasil,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2277,NaN,NaN,Não,NaN,NaN,NaN,NaN,NaN,{1A1FCA79-CCB2-F111-AAAB-002248E10B2D},NaT,...,NaN,NaN,Brasil,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
linhas_em_branco.info()

<class 'pandas.DataFrame'>
Index: 6035 entries, 1226 to 7262
Data columns (total 32 columns):
 #   Column                  Non-Null Count  Dtype        
---  ------                  --------------  -----        
 0   Inscrição Atual         0 non-null      str          
 1   cpf                     0 non-null      str          
 2   Tabulação Realizada     6035 non-null   str          
 3   Nome Completo           0 non-null      str          
 4   Nome                    0 non-null      str          
 5   Sobrenome               0 non-null      str          
 6   Email                   0 non-null      str          
 7   Celular                 0 non-null      str          
 8   ID Lead                 6035 non-null   str          
 9   Data Criação            0 non-null      datetime64[s]
 10  Idade                   0 non-null      str          
 11  Concurso Atual          0 non-null      str          
 12  Concurso de Origem      0 non-null      str          
 13  Data Nascimento 

In [38]:
print(f'quantidade de leads em branco: {linhas_em_branco.shape[0]}')

quantidade de leads em branco: 6035


In [39]:
linhas_em_branco = linhas_em_branco[['ID Lead', 'Nome Completo']]

In [40]:
inscricoes = inscricoes[['ID Lead', 'id_inscricao']]

In [41]:
df = linhas_em_branco.merge(inscricoes, how= 'left', on = 'ID Lead')

In [42]:
df.head()

,ID Lead,Nome Completo,id_inscricao
0,{C1793A09-CDB2-F111-AAAB-002248E10B2D},NaN,NaN
1,{C9793A09-CDB2-F111-AAAB-002248E10B2D},NaN,NaN
2,{D0793A09-CDB2-F111-AAAB-002248E10B2D},NaN,NaN
3,{A5793A09-CDB2-F111-AAAB-002248E10B2D},NaN,NaN
4,{AC793A09-CDB2-F111-AAAB-002248E10B2D},NaN,NaN


In [43]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6035 entries, 0 to 6034
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   ID Lead        6035 non-null   str  
 1   Nome Completo  0 non-null      str  
 2   id_inscricao   0 non-null      str  
dtypes: str(3)
memory usage: 367.7 KB


In [44]:
colunas = ['ID Lead','id_inscricao']

for coluna in colunas:
    df[coluna] = df[coluna].str.strip().str.lower()
    df[coluna] = df[coluna].str.replace('{', '').str.replace('}', '')

In [45]:
df['Nome Completo'] = df['Nome Completo'].str.strip().str.lower()

In [47]:
df

,ID Lead,Nome Completo,id_inscricao
0,c1793a09-cdb2-f111-aaab-002248e10b2d,NaN,NaN
1,c9793a09-cdb2-f111-aaab-002248e10b2d,NaN,NaN
2,d0793a09-cdb2-f111-aaab-002248e10b2d,NaN,NaN
3,a5793a09-cdb2-f111-aaab-002248e10b2d,NaN,NaN
4,ac793a09-cdb2-f111-aaab-002248e10b2d,NaN,NaN
...,...,...,...
6030,ce92567a-c8b2-f111-aaab-002248e10b2d,NaN,NaN
6031,d592567a-c8b2-f111-aaab-002248e10b2d,NaN,NaN
6032,ac92567a-c8b2-f111-aaab-002248e10b2d,NaN,NaN
6033,b392567a-c8b2-f111-aaab-002248e10b2d,NaN,NaN


In [48]:
df.to_csv('dataset.csv', index= False)